In [1]:
# Import von Bibliotheken
import urllib.request
import io,re
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
!pip install textstat
import textstat
!pip install simplemma
!pip install lexicalrichness
import simplemma
from simplemma import simple_tokenizer
from lexicalrichness import LexicalRichness
!pip install nltk
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords = stopwords.words('german')

  Using cached simplemma-1.2.0-py3-none-any.whl.metadata (24 kB)
Using cached simplemma-1.2.0-py3-none-any.whl (67.5 MB)
  Using cached lexicalrichness-0.5.1-py3-none-any.whl
  Using cached textblob-0.20.1-py3-none-any.whl.metadata (4.0 kB)
Using cached textblob-0.20.1-py3-none-any.whl (624 kB)


[nltk_data] Downloading package punkt to /home/haloh001/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/haloh001/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**Wahlprogramme**

**1. Teil: Anteil Wörter mit 14 oder mehr Buchstaben**

In [2]:
# Einlesen der Langwahlprogramme (aus JSON-Format in ein Wörterbuch)

# für CDU/CSU
with open("cdu-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    CDU_text_json = json.load(datei)

# für SPD
with open("spd-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    SPD_text_json = json.load(datei)

# für Die Linke
with open("linke-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    Linke_text_json = json.load(datei)

# für die AfD
with open("afd-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    AfD_text_json = json.load(datei)

# für Bündnis 90/Grüne
with open("gruene-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    Grüne_text_json = json.load(datei)

# für FDP
with open("fdp-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    FDP_text_json = json.load(datei)

In [3]:
# Überführen aller 'eigentlichen' Texte in eine Liste pro Partei, d.h. ohne Kapitelüberschriften

# Suchfunktion in den "Partei-Wörterbüchern"

def finde_gefilterte_texte(struktur):
    ergebnisse = []
    
    # Fall A: Wenn das aktuelle Element ein Dictionary ist
    if isinstance(struktur, dict):
        # Typ/Art auslesen (liefert None, wenn der Schlüssel fehlt)
        aktueller_typ = struktur.get('type') or struktur.get('art')            # nur Suchen in diesen Elementen
        
        # Prüfen, ob 'text' existiert und der gefundene Typ gültig ist
        if 'text' in struktur and aktueller_typ in ['paragraph', 'bullet', 'absatz']:   # nur Texte überführen mit diesem Typ
            ergebnisse.append(struktur['text'])
        
        # Tiefer in alle Werte schauen für eventuelle Verschachtelungen
        for wert in struktur.values():
            ergebnisse.extend(finde_gefilterte_texte(wert))
                
    # Fall B: Wenn das aktuelle Element eine Liste ist
    elif isinstance(struktur, list):
        for element in struktur:
            ergebnisse.extend(finde_gefilterte_texte(element))
            
    return ergebnisse

CDU_text = finde_gefilterte_texte(CDU_text_json)
SPD_text = finde_gefilterte_texte(SPD_text_json)
Linke_text = finde_gefilterte_texte(Linke_text_json)
Grüne_text = finde_gefilterte_texte(Grüne_text_json)
AfD_text = finde_gefilterte_texte(AfD_text_json)
FDP_text = finde_gefilterte_texte(FDP_text_json)

In [4]:
# Überführen aller 'eigentlichen' Texte in einen String pro Partei, d.h. ohne Kapitelüberschriften

# für CDU/CSU
CDU_text_str = [text if text.endswith('.') else text + '.' for text in CDU_text]   # ein "Punkt" wird gesetzt bei Bulletpoints
CDU_text = " ".join(CDU_text_str)

# für SPD
SPD_text_str = [text if text.endswith('.') else text + '.' for text in SPD_text]
SPD_text = " ".join(SPD_text_str)

# für Die Linke
Linke_text_str = [text if text.endswith('.') else text + '.' for text in Linke_text]
Linke_text = " ".join(Linke_text_str)

# für die AfD
AfD_text_str = [text if text.endswith('.') else text + '.' for text in AfD_text]
AfD_text = " ".join(AfD_text_str)

# für Bündnis 90/Grüne
Grüne_text_str = [text if text.endswith('.') else text + '.' for text in Grüne_text]
Grüne_text = " ".join(Grüne_text_str)

# für FDP
FDP_text_str = [text if text.endswith('.') else text + '.' for text in FDP_text]
FDP_text = " ".join(FDP_text_str)

In [5]:
# Extrahiert alle Wörter (ohne Punkt, Komma etc.)

CDU_WP_wortliste = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*', CDU_text)  # nach dem "+(?..." ist Ausdruck hinzugekommen
SPD_WP_wortliste = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*', SPD_text)
Linke_WP_wortliste = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*', Linke_text)
Grüne_WP_wortliste = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*', Grüne_text)
AfD_WP_wortliste = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*', AfD_text)
FDP_WP_wortliste = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*', FDP_text)

# Filtert Wörter mit mehr als 14 Zeichen
CDU_WP_wortliste_14 = [word for word in CDU_WP_wortliste if len(word) >= 14]
SPD_WP_wortliste_14 = [word for word in SPD_WP_wortliste if len(word) >= 14]
Linke_WP_wortliste_14 = [word for word in Linke_WP_wortliste if len(word) >= 14]
Grüne_WP_wortliste_14 = [word for word in Grüne_WP_wortliste if len(word) >= 14]
AfD_WP_wortliste_14 = [word for word in AfD_WP_wortliste if len(word) >= 14]
FDP_WP_wortliste_14 = [word for word in FDP_WP_wortliste if len(word) >= 14]

# Verhältnisermittlung
CDU_14er_ratio = round((len(CDU_WP_wortliste_14) / len(CDU_WP_wortliste))*100,2)
SPD_14er_ratio = round((len(SPD_WP_wortliste_14) / len(SPD_WP_wortliste))*100,2)
Linke_14er_ratio = round((len(Linke_WP_wortliste_14) / len(Linke_WP_wortliste))*100,2)
Grüne_14er_ratio = round((len(Grüne_WP_wortliste_14) / len(Grüne_WP_wortliste))*100,2)
AfD_14er_ratio = round((len(AfD_WP_wortliste_14) / len(AfD_WP_wortliste))*100,2)
FDP_14er_ratio = round((len(FDP_WP_wortliste_14) / len(FDP_WP_wortliste))*100,2)

**2. Teil: durchschnittliche Wortlänge (für Wahlprogramme)**


In [6]:
#  durchschnittliche Länge auf 2 Nachkommenstellen gerundet
CDU_Schnitt_Wortlänge_WP = round(sum(len(wort) for wort in CDU_WP_wortliste) / len(CDU_WP_wortliste),2)
SPD_Schnitt_Wortlänge_WP = round(sum(len(wort) for wort in SPD_WP_wortliste) / len(SPD_WP_wortliste),2)
Linke_Schnitt_Wortlänge_WP = round(sum(len(wort) for wort in Linke_WP_wortliste) / len(Linke_WP_wortliste),2)
Grüne_Schnitt_Wortlänge_WP = round(sum(len(wort) for wort in Grüne_WP_wortliste) / len(Grüne_WP_wortliste),2)
AfD_Schnitt_Wortlänge_WP = round(sum(len(wort) for wort in AfD_WP_wortliste) / len(AfD_WP_wortliste),2)
FDP_Schnitt_Wortlänge_WP = round(sum(len(wort) for wort in FDP_WP_wortliste) / len(FDP_WP_wortliste),2)

print("#####")
print(CDU_Schnitt_Wortlänge_WP)
print(SPD_Schnitt_Wortlänge_WP)
print(Linke_Schnitt_Wortlänge_WP)
print(Grüne_Schnitt_Wortlänge_WP)
print(AfD_Schnitt_Wortlänge_WP)
print(FDP_Schnitt_Wortlänge_WP)

#####
6.82
6.73
6.77
6.77
7.02
6.9


**3. Teil: Schachtelsätze ("Hypotaxe") bei den Wahlprogrammen**

In [7]:
def split_sentences(text):
    # Häufige deutsche Abkürzungen (erweiterbar!)
    abbreviations = [
        "z.B.", "u.a.", "bzw.", "d.h.", "etc.", "Dr.", "Prof.", "Nr.", "ca.", "vgl."
    ]

    # 1. Abkürzungen maskieren
    abbr_map = {}
    for i, abbr in enumerate(abbreviations):
        placeholder = f"__ABBR_{i}__"
        text = text.replace(abbr, placeholder)
        abbr_map[placeholder] = abbr

    # 2. Satztrennung
    sentences = re.split(r'(?<=[.!?])\s+', text)

    # 3. Abkürzungen wiederherstellen
    cleaned_sentences = []
    for s in sentences:
        for placeholder, abbr in abbr_map.items():
            s = s.replace(placeholder, abbr)
        s = s.strip()
        if s:
            cleaned_sentences.append(s)

    return cleaned_sentences

In [8]:
# Nutzt Regular Expressions aus Vorzelle, um Sätze zu extrahieren
CDU_sätze = split_sentences(CDU_text)
SPD_sätze = split_sentences(SPD_text)
Linke_sätze = split_sentences(Linke_text)
Grüne_sätze = split_sentences(Grüne_text)
AfD_sätze = split_sentences(AfD_text)
FDP_sätze = split_sentences(FDP_text)

In [9]:
# Sätze löschen, die nur Zahlen und Sonderzeichen enthalten
import string

def filter_list(liste):
  sonderzeichen = re.escape(string.punctuation)
  muster = f"^[0-9\\s{sonderzeichen}]+$"
  return [s for s in liste if not re.match(muster, s)]

In [10]:
CDU_sätze_bereinigt = filter_list(CDU_sätze)
SPD_sätze_bereinigt = filter_list(SPD_sätze)
Linke_sätze_bereinigt = filter_list(Linke_sätze)
Grüne_sätze_bereinigt = filter_list(Grüne_sätze)
AfD_sätze_bereinigt = filter_list(AfD_sätze)
FDP_sätze_bereinigt = filter_list(FDP_sätze)

In [11]:
# Sätze löschen, die nur ein Zeichen enthält
def filter_list_2(lst):
    return [x for x in lst if len(x) > 1]

In [12]:
CDU_sätze_2 = filter_list_2(CDU_sätze_bereinigt)
SPD_sätze_2 = filter_list_2(SPD_sätze_bereinigt)
Linke_sätze_2 = filter_list_2(Linke_sätze_bereinigt)
Grüne_sätze_2 = filter_list_2(Grüne_sätze_bereinigt)
AfD_sätze_2 = filter_list_2(AfD_sätze_bereinigt)
FDP_sätze_2 = filter_list_2(FDP_sätze_bereinigt)

CDU_sätze_end = " ".join(CDU_sätze_2)
SPD_sätze_end = " ".join(SPD_sätze_2)
Linke_sätze_end = " ".join(Linke_sätze_2)
Grüne_sätze_end = " ".join(Grüne_sätze_2)
AfD_sätze_end = " ".join(AfD_sätze_2)
FDP_sätze_end = " ".join(FDP_sätze_2)

In [13]:
# Nutzt Regular Expressions, um bei . ! oder ? zu trennen
# Das [?!.] definiert eine Menge von Zeichen
CDU_sätze  = re.split(r'[?!.]', CDU_sätze_end)
CDU_sätze = [s.strip() for s in CDU_sätze if s.strip()]

SPD_sätze  = re.split(r'[?!.]', SPD_sätze_end)
SPD_sätze = [s.strip() for s in SPD_sätze if s.strip()]

Linke_sätze  = re.split(r'[?!.]', Linke_sätze_end)
Linke_sätze = [s.strip() for s in Linke_sätze if s.strip()]

Grüne_sätze  = re.split(r'[?!.]', Grüne_sätze_end)
Grüne_sätze = [s.strip() for s in Grüne_sätze if s.strip()]

AfD_sätze  = re.split(r'[?!.]', AfD_sätze_end)
AfD_sätze = [s.strip() for s in AfD_sätze if s.strip()]

FDP_sätze  = re.split(r'[?!.]', FDP_sätze_end)
FDP_sätze = [s.strip() for s in FDP_sätze if s.strip()]

In [14]:
# Anzahl Sätze mit über 2 oder mehr Kommata
CDU_anzahl_schachtelsätze = len([s for s in CDU_sätze if s.count(',') >= 2])
SPD_anzahl_schachtelsätze = len([s for s in SPD_sätze if s.count(',') >= 2])
Linke_anzahl_schachtelsätze = len([s for s in Linke_sätze if s.count(',') >= 2])
Grüne_anzahl_schachtelsätze = len([s for s in Grüne_sätze if s.count(',') >= 2])
AfD_anzahl_schachtelsätze = len([s for s in AfD_sätze if s.count(',') >= 2])
FDP_anzahl_schachtelsätze = len([s for s in FDP_sätze if s.count(',') >= 2])

In [15]:
# Verhältnisermittlung
CDU_schachtel_ratio = round(CDU_anzahl_schachtelsätze / (len(CDU_sätze))*100,2)
SPD_schachtel_ratio = round(SPD_anzahl_schachtelsätze / (len(SPD_sätze))*100,2)
Linke_schachtel_ratio = round(Linke_anzahl_schachtelsätze / (len(Linke_sätze))*100,2)
Grüne_schachtel_ratio = round(Grüne_anzahl_schachtelsätze / (len(Grüne_sätze))*100,2)
AfD_schachtel_ratio = round(AfD_anzahl_schachtelsätze / (len(AfD_sätze))*100,2)
FDP_schachtel_ratio = round(FDP_anzahl_schachtelsätze / (len(FDP_sätze))*100,2)

**Bundestagsreden**

In [16]:
import json
import urllib.request, urllib.parse, urllib.error
stammdaten = ("MDB_STAMMDATEN.XML")  # mit den Parteizugehörigkeiten
from bs4 import BeautifulSoup
with open(stammdaten, "r", encoding="utf-8") as f:
  	soup2 = BeautifulSoup(f, "xml")

redner_id_mdb = dict()
for mdb in soup2.find_all("MDB"):
	id_tag = mdb.find("ID")
	party_tag = mdb.find("PARTEI_KURZ")
	mdb_id = id_tag.text if id_tag else None
	party = party_tag.text if party_tag else None
	redner_id_mdb[mdb_id] = party

# da wo keine Parteizugehörigkeit aus den Stammdaten ersichtlich, manuelles Hinzufügen diverser Nummern
redner_id_mdb["11002735"]  # MERZ
redner_id_mdb["999990151"] = "SPD"
redner_id_mdb["999990133"] = "SPD"
redner_id_mdb["999990074"] = "SPD"
redner_id_mdb["11005217 999990074"] = "SPD"
redner_id_mdb["999990119"] = "SPD"
redner_id_mdb["999990149"] = "SPD"
redner_id_mdb["999990080"] = "parteilos"
redner_id_mdb["999990142"] = "parteilos"
redner_id_mdb["999990154"] = "parteilos"
redner_id_mdb["999990152"] = "CDU"
redner_id_mdb["999990153"] = "CDU"
redner_id_mdb["999990150"] = "CDU"
redner_id_mdb["999990141"] = "CSU"
redner_id_mdb["999990193"] = "SPD"
redner_id_mdb["999990093"] = "SPD"
redner_id_mdb["999990120"] = "SPD"
redner_id_mdb["999990129"] = "SPD"
redner_id_mdb["999990145"] = "SPD"
redner_id_mdb["999990144"] = "CDU"
redner_id_mdb["999990125"] = "CDU"
redner_id_mdb["999990147"] = "CSU"
redner_id_mdb["999990146"] = "SPD"
redner_id_mdb["999990121"] = "SPD"
redner_id_mdb["999990148"] = "SPD"
redner_id_mdb["999990122"] = "BÜNDNIS 90/DIE GRÜNEN"
redner_id_mdb["999990123"] = "DIE LINKE"
redner_id_mdb["999990148"] = "SPD"
redner_id_mdb["999990122"] = "BÜNDNIS 90/DIE GRÜNEN"
redner_id_mdb["999990123"] = "DIE LINKE"
redner_id_mdb["999990124"] = "SPD"
redner_id_mdb["999990078"] = "FDP"
redner_id_mdb["999990082"] = "SPD"

In [17]:
# Import der Bundestagsreden via gültigem API-Key per Juli 2026 für die 5 Betrachtungszeiträume
api = "R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ"

# Download für Zeitraum Nr. 1 und Umwandlung in json-Format
html_zeitraum_1 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2022-01-12&f.datum.end=2022-03-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_1 = json.loads(html_zeitraum_1)

# Download für Zeitraum Nr. 2 und Umwandlung in json-Format
html_zeitraum_2 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2023-10-01&f.datum.end=2023-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_2 = json.loads(html_zeitraum_2)

# Download für Zeitraum Nr. 3 und Umwandlung in json-Format
html_zeitraum_3 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2024-10-01&f.datum.end=2024-12-20&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_3 = json.loads(html_zeitraum_3)

# Download für Zeitraum Nr. 4 und Umwandlung in json-Format
html_zeitraum_4 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-01-01&f.datum.end=2025-02-23&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_4 = json.loads(html_zeitraum_4)

# Download für Zeitraum Nr. 5 und Umwandlung in json-Format
html_zeitraum_5 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-05-01&f.datum.end=2025-07-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_5 = json.loads(html_zeitraum_5)

In [18]:
# Import der Bundestagsreden via gültigem API-Key für die 5 Betrachtungszeiträume
# API-Key gültig bis zunächst Ende Mai 2027
api = "R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ"

# Eingrenzung Zeitraum Nr. 1, Umwandlung in json-Format & Extraktion der XML-URLs (=Protokolle) aus dem Dictionary
html_zeitraum_1 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2022-01-01&f.datum.end=2022-03-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_1 = json.loads(html_zeitraum_1)
# Erfassen der XML-URLs in einer Liste
data_zeitraum_1_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_1["documents"]
    if "xml_url" in doc["fundstelle"]
]   # 17 Protokolle

# Eingrenzung Zeitraum Nr. 2...
html_zeitraum_2 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2023-10-01&f.datum.end=2023-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_2 = json.loads(html_zeitraum_2)
data_zeitraum_2_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_2["documents"]
    if "xml_url" in doc["fundstelle"]
]  # 19 Protokolle

# Eingrenzung Zeitraum Nr. 3...
html_zeitraum_3 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2024-10-01&f.datum.end=2024-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_3 = json.loads(html_zeitraum_3)
data_zeitraum_3_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_3["documents"]
    if "xml_url" in doc["fundstelle"]
]   # 19 Protokolle

# Eingrenzung Zeitraum Nr. 4...
html_zeitraum_4 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-01-01&f.datum.end=2025-02-23&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_4 = json.loads(html_zeitraum_4)
data_zeitraum_4_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_4["documents"]
    if "xml_url" in doc["fundstelle"]
]    # 4 Protokolle

# Eingrenzung Zeitraum Nr. 5...
html_zeitraum_5 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-05-01&f.datum.end=2025-07-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_5 = json.loads(html_zeitraum_5)
data_zeitraum_5_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_5["documents"]
    if "xml_url" in doc["fundstelle"]   
]    # 18 Protokolle

# Texte pro Partei und Zeitraum
text_liste_CDU_1 = []
text_liste_CDU_2 = []
text_liste_CDU_3 = []
text_liste_CDU_4 = []
text_liste_CDU_5 = []

text_liste_SPD_1 = []
text_liste_SPD_2 = []
text_liste_SPD_3 = []
text_liste_SPD_4 = []
text_liste_SPD_5 = []

text_liste_FDP_1 = []
text_liste_FDP_2 = []
text_liste_FDP_3 = []
text_liste_FDP_4 = []
text_liste_FDP_5 = []

text_liste_Grüne_1 = []
text_liste_Grüne_2 = []
text_liste_Grüne_3 = []
text_liste_Grüne_4 = []
text_liste_Grüne_5 = []

text_liste_Linke_1 = []
text_liste_Linke_2 = []
text_liste_Linke_3 = []
text_liste_Linke_4 = []
text_liste_Linke_5 = []

text_liste_AfD_1 = []
text_liste_AfD_2 = []
text_liste_AfD_3 = []
text_liste_AfD_4 = []
text_liste_AfD_5 = []

for i in data_zeitraum_1_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_1.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_1.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_1.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_1.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_1.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_1.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_1.append(p)

for i in data_zeitraum_2_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_2.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_2.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_2.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_2.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_2.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_2.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_2.append(p)

for i in data_zeitraum_3_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_3.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_3.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_3.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_3.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_3.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_3.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_3.append(p)

for i in data_zeitraum_4_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_4.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_4.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_4.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_4.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_4.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_4.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_4.append(p)

for i in data_zeitraum_5_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_5.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_5.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_5.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_5.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_5.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_5.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_5.append(p)


In [19]:
# Bereinigung der Texte um Texte der Bundestagspräsidentin und sonstigen Nicht-Rede-Elementen

def bereinige_text_liste(liste):
    bereinigte_liste = []
    for i in liste:
        i = str(i[1:]).replace('<p klasse="J_1">', "")
        i = i.replace('<p klasse="J">', "")
        i = i.replace('<p klasse="O">', "")
        i = i.replace("<p klasse=", "")
        i = re.sub(r'-\s+', '', i)    # neu
        i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)  # neu
        i = i.replace("p, ", " ")
        i = i.replace(".p, p", ".")
        i = i.replace(":p, ",": ")
        i = re.sub(r".*?hat als Nächstes das Wort für.*?\.", "", i)   # dies sind Texte von der Bundestagspräsidentin und müssen aussortiert werden
        i = re.sub(r'Das\s+Wort\s+hat\s+nun\s+für\s+die\s+Fraktion\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Ich\s+erteile\s+das\s+Wort\s+als\s+Nächstes\s+[\w\s. -ÄÖÜäöüß]+[\.!?]', '', i)
        i = re.sub(r'Für\s+die\s+Fraktion\s+[\w\s.-]+\s+hat\s+nun\s+das\s+Wort\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Für\s+die\s+Fraktion\s+[\w\s.-]+\s+spricht\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Für\s+die\s+[\w\s.-]+\s+hat\s+nun\s+[\w\s.-]+\s+das\s+Wort[\.!?]', '', i)
        i = re.sub(r'^[A-Z][a-zßäöü]+ [A-Z][a-zßäöü]+, kommen Sie bitte zu Ihrer Rede\.', '', i)
        i = re.sub(r"Das Wort für .*? hat .*?\.", "", i)
        i = i.replace("Kommen Sie bitte zum Ende Ihrer Rede.", "")
        i = i.replace("Kommen Sie bitte zum Ende. ", "")
        i = i.replace("Es ist ihre erste Rede.", "")
        i = re.sub(r'rednerredner\s+id\d+[\w\s.-]*:', '', i)
        i = i.replace(", kommen Sie bitte zum Ende Ihrer Rede.p", "")
        i = i.replace(", kommen Sie bitte zum Schluss.", "")
        i = i.replace(", kommen Sie bitte zu Ihrer Rede.", "")
        i = i.replace("Sehr geehrter Herr Botschafter, ", "")
        i = re.sub(r'Die\s+nächste\s+Rednerin\s+ist\s+[\w\s.-]+\s+für\s+die\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Der\s+nächste\s+Redner\s+ist\s+[\w\s.-]+\s+für\s+die\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'[D|d]er nächste Redner in der Debatte ist [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)?\.?', '', i)
        i = re.sub(r'[D|d]ie nächste Rednerin in der Debatte ist [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)?\.?', '', i)
        i = re.sub(r'rednerredner\s+[\w\s. -ÄÖÜäöüß]+:', '', i)
        i = re.sub(r'[F|f]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+-Fraktion erhält das Wort [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?)?\.?', '', i)
        i = re.sub(r'Als\s+Nächstes\s+spricht\s+[\w\s. -ÄÖÜäöüß]+[\.!?]', '', i)
        i = re.sub(r'[I|i]ch darf für die Fraktion (?:[A-Zßäöüa-z\s\/äöü]+) [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? aufrufen\.?', '', i)
        i = re.sub(r'[I|i]ch erteile das Wort für die nächste Rede (?:[A-ZßäöüÄÖÜa-z\s\/äöü]+) [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?)?\.?', '', i)
        i = re.sub(r'[I|i]ch erteile als Nächstes das Wort de[mr] Abgeordneten [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)* für die [A-Za-zßäöüÄÖÜ\s\/]+(?:\.)?', '', i)
        i = re.sub(r'[D|d]ie nächste Rede hält [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)* für die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[I|i]ch darf aufrufen für die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[D|d]ie nächste Rede hält [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)*(?:\.)?', '', i)
        i = re.sub(r'[A[Aa]ls nächste(?:r)? (?:Rednerin|Redner) hat [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort\.?', '', i)
        i = re.sub(r'[E[Ee]benfalls zur ersten Rede erteile ich das Wort [A-ZßäöüÄÖÜa-z\s\-\/\.]+\.?', '', i)
        i = re.sub(r'[Ii]ch darf [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort erteilen\.?', '', i)
        i = re.sub(r'[Ii]ch darf [A-ZßäöüÄÖÜa-z\s\-\/\.]+ aufrufen\.?', '', i)
        i = re.sub(r'[Ii]ch erteile das Wort [A-ZßäöüÄÖÜa-z\s\-\/\.]+\.?', '', i)
        i = re.sub(r'[Vv]ielen Dank und Gratulation zu Ihrer ersten Rede, (?:Frau|Herr) [A-ZßäöüÄÖÜa-z\s\-\/\.]+(?:\.)?', '', i)
        i = re.sub(r'[Ff]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+ das Wort zu seiner ersten Rede\.', '', i)
        i = re.sub(r'[Ff]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+ das Wort zu ihrer ersten Rede\.', '', i)
        i = re.sub(r'[Dd]er nächste Redner in der Debatte: für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Dd]ie nächste Rednerin in der Debatte: für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Dd]ann rufe ich [A-ZßäöüÄÖÜa-z\s\-\/\.]+ in der Debatte auf: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+)? für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Zz]u (?:seiner|ihrer) ersten Rede hat nun [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort\.?', '', i)
        i = re.sub(r'von der [A-Za-zßäöüÄÖÜ\s\-\/0-9]+, für (?:den|die) es hier die erste Rede ist\.?', '', i)
        i = i.replace("Vielen Dank Ihnen. ", "")
        i = i.removesuffix('p')
        i = re.sub(r"\s+", " ", i)
        i = i.replace("\xa0", "")
        i = i.replace("</p>", "")
        i = i.replace("</p", "")
        i = i.replace("p, p, ", "")
        i = i.lstrip()
        i = i.rstrip(" ")
        bereinigte_liste.append(i)
    return bereinigte_liste

# Zeitraum 1
text_liste_CDU_1_clean = bereinige_text_liste(text_liste_CDU_1)
text_liste_SPD_1_clean = bereinige_text_liste(text_liste_SPD_1)
text_liste_Linke_1_clean = bereinige_text_liste(text_liste_Linke_1)
text_liste_AfD_1_clean = bereinige_text_liste(text_liste_AfD_1)
text_liste_Grüne_1_clean = bereinige_text_liste(text_liste_Grüne_1)
text_liste_FDP_1_clean = bereinige_text_liste(text_liste_FDP_1)

# Zeitraum 2
text_liste_CDU_2_clean = bereinige_text_liste(text_liste_CDU_2)
text_liste_SPD_2_clean = bereinige_text_liste(text_liste_SPD_2)
text_liste_Linke_2_clean = bereinige_text_liste(text_liste_Linke_2)
text_liste_AfD_2_clean = bereinige_text_liste(text_liste_AfD_2)
text_liste_Grüne_2_clean = bereinige_text_liste(text_liste_Grüne_2)
text_liste_FDP_2_clean = bereinige_text_liste(text_liste_FDP_2)

# Zeitraum 3
text_liste_CDU_3_clean = bereinige_text_liste(text_liste_CDU_3)
text_liste_SPD_3_clean = bereinige_text_liste(text_liste_SPD_3)
text_liste_Linke_3_clean = bereinige_text_liste(text_liste_Linke_3)
text_liste_AfD_3_clean = bereinige_text_liste(text_liste_AfD_3)
text_liste_Grüne_3_clean = bereinige_text_liste(text_liste_Grüne_3)
text_liste_FDP_3_clean = bereinige_text_liste(text_liste_FDP_3)

# Zeitraum 4
text_liste_CDU_4_clean = bereinige_text_liste(text_liste_CDU_4)
text_liste_SPD_4_clean = bereinige_text_liste(text_liste_SPD_4)
text_liste_Linke_4_clean = bereinige_text_liste(text_liste_Linke_4)
text_liste_AfD_4_clean = bereinige_text_liste(text_liste_AfD_4)
text_liste_Grüne_4_clean = bereinige_text_liste(text_liste_Grüne_4)
text_liste_FDP_4_clean = bereinige_text_liste(text_liste_FDP_4)

# Zeitraum 5
text_liste_CDU_5_clean = bereinige_text_liste(text_liste_CDU_5)
text_liste_SPD_5_clean = bereinige_text_liste(text_liste_SPD_5)
text_liste_Linke_5_clean = bereinige_text_liste(text_liste_Linke_5)
text_liste_AfD_5_clean = bereinige_text_liste(text_liste_AfD_5)
text_liste_Grüne_5_clean = bereinige_text_liste(text_liste_Grüne_5)
text_liste_FDP_5_clean = bereinige_text_liste(text_liste_FDP_5)

In [20]:
# aus jeder Liste einen großen String pro Partei und Betrachtungszeitraum
text_liste_CDU_1_str = "".join(text_liste_CDU_1_clean)
text_liste_SPD_1_str = "".join(text_liste_SPD_1_clean)
text_liste_AfD_1_str = "".join(text_liste_AfD_1_clean)
text_liste_Linke_1_str = "".join(text_liste_Linke_1_clean)
text_liste_Grüne_1_str = "".join(text_liste_Grüne_1_clean)
text_liste_FDP_1_str = "".join(text_liste_FDP_1_clean)

text_liste_CDU_2_str = "".join(text_liste_CDU_2_clean)
text_liste_SPD_2_str = "".join(text_liste_SPD_2_clean)
text_liste_AfD_2_str = "".join(text_liste_AfD_2_clean)
text_liste_Linke_2_str = "".join(text_liste_Linke_2_clean)
text_liste_Grüne_2_str = "".join(text_liste_Grüne_2_clean)
text_liste_FDP_2_str = "".join(text_liste_FDP_2_clean)

text_liste_CDU_3_str = "".join(text_liste_CDU_3_clean)
text_liste_SPD_3_str = "".join(text_liste_SPD_3_clean)
text_liste_AfD_3_str = "".join(text_liste_AfD_3_clean)
text_liste_Linke_3_str = "".join(text_liste_Linke_3_clean)
text_liste_Grüne_3_str = "".join(text_liste_Grüne_3_clean)
text_liste_FDP_3_str = "".join(text_liste_FDP_3_clean)

text_liste_CDU_4_str = "".join(text_liste_CDU_4_clean)
text_liste_SPD_4_str = "".join(text_liste_SPD_4_clean)
text_liste_AfD_4_str = "".join(text_liste_AfD_4_clean)
text_liste_Linke_4_str = "".join(text_liste_Linke_4_clean)
text_liste_Grüne_4_str = "".join(text_liste_Grüne_4_clean)
text_liste_FDP_4_str = "".join(text_liste_FDP_4_clean)

text_liste_CDU_5_str = "".join(text_liste_CDU_5_clean)
text_liste_SPD_5_str = "".join(text_liste_SPD_5_clean)
text_liste_AfD_5_str = "".join(text_liste_AfD_5_clean)
text_liste_Linke_5_str = "".join(text_liste_Linke_5_clean)
text_liste_Grüne_5_str = "".join(text_liste_Grüne_5_clean)
text_liste_FDP_5_str = "".join(text_liste_FDP_5_clean)

**1. Teil: Anteil Wörter mit 14 oder mehr Buchstaben (für Bundestagsreden)**

In [21]:
# Extrahiert alle Wörter (ohne Punkt, Komma etc.)
CDU_BT_wortliste_1 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_CDU_1_str)
CDU_BT_wortliste_2 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_CDU_2_str)
CDU_BT_wortliste_3 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_CDU_3_str)
CDU_BT_wortliste_4 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_CDU_4_str)
CDU_BT_wortliste_5 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_CDU_5_str)

SPD_BT_wortliste_1 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_SPD_1_str)
SPD_BT_wortliste_2 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_SPD_2_str)
SPD_BT_wortliste_3 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_SPD_3_str)
SPD_BT_wortliste_4 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_SPD_4_str)
SPD_BT_wortliste_5 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_SPD_5_str)

Linke_BT_wortliste_1 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Linke_1_str)
Linke_BT_wortliste_2 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Linke_2_str)
Linke_BT_wortliste_3 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Linke_3_str)
Linke_BT_wortliste_4 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Linke_4_str)
Linke_BT_wortliste_5 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Linke_5_str)

Grüne_BT_wortliste_1 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Grüne_1_str)
Grüne_BT_wortliste_2 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Grüne_2_str)
Grüne_BT_wortliste_3 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Grüne_3_str)
Grüne_BT_wortliste_4 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Grüne_4_str)
Grüne_BT_wortliste_5 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_Grüne_5_str)

AfD_BT_wortliste_1 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_AfD_1_str)
AfD_BT_wortliste_2 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_AfD_2_str)
AfD_BT_wortliste_3 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_AfD_3_str)
AfD_BT_wortliste_4 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_AfD_4_str)
AfD_BT_wortliste_5 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_AfD_5_str)

FDP_BT_wortliste_1 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_FDP_1_str)
FDP_BT_wortliste_2 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_FDP_2_str)
FDP_BT_wortliste_3 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_FDP_3_str)
FDP_BT_wortliste_4 = re.findall(r'[a-zA-ZäöüÄÖÜß]+(?:-[a-zA-ZäöüÄÖÜß]+)*',  text_liste_FDP_4_str)

# Filtert Wörter mit mehr als 14 Zeichen
CDU_BT_wortliste_14_1 = [word for word in CDU_BT_wortliste_1 if len(word) >= 14]
CDU_BT_wortliste_14_2 = [word for word in CDU_BT_wortliste_2 if len(word) >= 14]
CDU_BT_wortliste_14_3 = [word for word in CDU_BT_wortliste_3 if len(word) >= 14]
CDU_BT_wortliste_14_4 = [word for word in CDU_BT_wortliste_4 if len(word) >= 14]
CDU_BT_wortliste_14_5 = [word for word in CDU_BT_wortliste_5 if len(word) >= 14]

SPD_BT_wortliste_14_1 = [word for word in SPD_BT_wortliste_1 if len(word) >= 14]
SPD_BT_wortliste_14_2 = [word for word in SPD_BT_wortliste_2 if len(word) >= 14]
SPD_BT_wortliste_14_3 = [word for word in SPD_BT_wortliste_3 if len(word) >= 14]
SPD_BT_wortliste_14_4 = [word for word in SPD_BT_wortliste_4 if len(word) >= 14]
SPD_BT_wortliste_14_5 = [word for word in SPD_BT_wortliste_5 if len(word) >= 14]

Linke_BT_wortliste_14_1 = [word for word in Linke_BT_wortliste_1 if len(word) >= 14]
Linke_BT_wortliste_14_2 = [word for word in Linke_BT_wortliste_2 if len(word) >= 14]
Linke_BT_wortliste_14_3 = [word for word in Linke_BT_wortliste_3 if len(word) >= 14]
Linke_BT_wortliste_14_4 = [word for word in Linke_BT_wortliste_4 if len(word) >= 14]
Linke_BT_wortliste_14_5 = [word for word in Linke_BT_wortliste_5 if len(word) >= 14]

Grüne_BT_wortliste_14_1 = [word for word in Grüne_BT_wortliste_1 if len(word) >= 14]
Grüne_BT_wortliste_14_2 = [word for word in Grüne_BT_wortliste_2 if len(word) >= 14]
Grüne_BT_wortliste_14_3 = [word for word in Grüne_BT_wortliste_3 if len(word) >= 14]
Grüne_BT_wortliste_14_4 = [word for word in Grüne_BT_wortliste_4 if len(word) >= 14]
Grüne_BT_wortliste_14_5 = [word for word in Grüne_BT_wortliste_5 if len(word) >= 14]

AfD_BT_wortliste_14_1 = [word for word in AfD_BT_wortliste_1 if len(word) >= 14]
AfD_BT_wortliste_14_2 = [word for word in AfD_BT_wortliste_2 if len(word) >= 14]
AfD_BT_wortliste_14_3 = [word for word in AfD_BT_wortliste_3 if len(word) >= 14]
AfD_BT_wortliste_14_4 = [word for word in AfD_BT_wortliste_4 if len(word) >= 14]
AfD_BT_wortliste_14_5 = [word for word in AfD_BT_wortliste_5 if len(word) >= 14]

FDP_BT_wortliste_14_1 = [word for word in FDP_BT_wortliste_1 if len(word) >= 14]
FDP_BT_wortliste_14_2 = [word for word in FDP_BT_wortliste_2 if len(word) >= 14]
FDP_BT_wortliste_14_3 = [word for word in FDP_BT_wortliste_3 if len(word) >= 14]
FDP_BT_wortliste_14_4 = [word for word in FDP_BT_wortliste_4 if len(word) >= 14]

# Verhältnisermittlung
CDU_14er_ratio_BT_1 = round((len(CDU_BT_wortliste_14_1) / len(CDU_BT_wortliste_1))*100,2)
CDU_14er_ratio_BT_2 = round((len(CDU_BT_wortliste_14_2) / len(CDU_BT_wortliste_2))*100,2)
CDU_14er_ratio_BT_3 = round((len(CDU_BT_wortliste_14_3) / len(CDU_BT_wortliste_3))*100,2)
CDU_14er_ratio_BT_4 = round((len(CDU_BT_wortliste_14_4) / len(CDU_BT_wortliste_4))*100,2)
CDU_14er_ratio_BT_5 = round((len(CDU_BT_wortliste_14_5) / len(CDU_BT_wortliste_5))*100,2)

SPD_14er_ratio_BT_1 = round((len(SPD_BT_wortliste_14_1) / len(SPD_BT_wortliste_1))*100,2)
SPD_14er_ratio_BT_2 = round((len(SPD_BT_wortliste_14_2) / len(SPD_BT_wortliste_2))*100,2)
SPD_14er_ratio_BT_3 = round((len(SPD_BT_wortliste_14_3) / len(SPD_BT_wortliste_3))*100,2)
SPD_14er_ratio_BT_4 = round((len(SPD_BT_wortliste_14_4) / len(SPD_BT_wortliste_4))*100,2)
SPD_14er_ratio_BT_5 = round((len(SPD_BT_wortliste_14_5) / len(SPD_BT_wortliste_5))*100,2)

Linke_14er_ratio_BT_1 = round((len(Linke_BT_wortliste_14_1) / len(Linke_BT_wortliste_1))*100,2)
Linke_14er_ratio_BT_2 = round((len(Linke_BT_wortliste_14_2) / len(Linke_BT_wortliste_2))*100,2)
Linke_14er_ratio_BT_3 = round((len(Linke_BT_wortliste_14_3) / len(Linke_BT_wortliste_3))*100,2)
Linke_14er_ratio_BT_4 = round((len(Linke_BT_wortliste_14_4) / len(Linke_BT_wortliste_4))*100,2)
Linke_14er_ratio_BT_5 = round((len(Linke_BT_wortliste_14_5) / len(Linke_BT_wortliste_5))*100,2)

Grüne_14er_ratio_BT_1 = round((len(Grüne_BT_wortliste_14_1) / len(Grüne_BT_wortliste_1))*100,2)
Grüne_14er_ratio_BT_2 = round((len(Grüne_BT_wortliste_14_2) / len(Grüne_BT_wortliste_2))*100,2)
Grüne_14er_ratio_BT_3 = round((len(Grüne_BT_wortliste_14_3) / len(Grüne_BT_wortliste_3))*100,2)
Grüne_14er_ratio_BT_4 = round((len(Grüne_BT_wortliste_14_4) / len(Grüne_BT_wortliste_4))*100,2)
Grüne_14er_ratio_BT_5 = round((len(Grüne_BT_wortliste_14_5) / len(Grüne_BT_wortliste_5))*100,2)

AfD_14er_ratio_BT_1 = round((len(AfD_BT_wortliste_14_1) / len(AfD_BT_wortliste_1))*100,2)
AfD_14er_ratio_BT_2 = round((len(AfD_BT_wortliste_14_2) / len(AfD_BT_wortliste_2))*100,2)
AfD_14er_ratio_BT_3 = round((len(AfD_BT_wortliste_14_3) / len(AfD_BT_wortliste_3))*100,2)
AfD_14er_ratio_BT_4 = round((len(AfD_BT_wortliste_14_4) / len(AfD_BT_wortliste_4))*100,2)
AfD_14er_ratio_BT_5 = round((len(AfD_BT_wortliste_14_5) / len(AfD_BT_wortliste_5))*100,2)

FDP_14er_ratio_BT_1 = round((len(FDP_BT_wortliste_14_1) / len(FDP_BT_wortliste_1))*100,2)
FDP_14er_ratio_BT_2 = round((len(FDP_BT_wortliste_14_2) / len(FDP_BT_wortliste_2))*100,2)
FDP_14er_ratio_BT_3 = round((len(FDP_BT_wortliste_14_3) / len(FDP_BT_wortliste_3))*100,2)
FDP_14er_ratio_BT_4 = round((len(FDP_BT_wortliste_14_4) / len(FDP_BT_wortliste_4))*100,2)

**2. Teil: durchschnittliche Wortlänge (für Bundestagsreden)**

In [22]:
# durchschnittliche Länge
CDU_Schnitt_Wortlänge_BT_1 = sum(len(wort) for wort in CDU_BT_wortliste_1) / len(CDU_BT_wortliste_1)
CDU_Schnitt_Wortlänge_BT_2 = sum(len(wort) for wort in CDU_BT_wortliste_2) / len(CDU_BT_wortliste_2)
CDU_Schnitt_Wortlänge_BT_3 = sum(len(wort) for wort in CDU_BT_wortliste_3) / len(CDU_BT_wortliste_3)
CDU_Schnitt_Wortlänge_BT_4 = sum(len(wort) for wort in CDU_BT_wortliste_4) / len(CDU_BT_wortliste_4)
CDU_Schnitt_Wortlänge_BT_5 = sum(len(wort) for wort in CDU_BT_wortliste_5) / len(CDU_BT_wortliste_5)

SPD_Schnitt_Wortlänge_BT_1 = sum(len(wort) for wort in SPD_BT_wortliste_1) / len(SPD_BT_wortliste_1)
SPD_Schnitt_Wortlänge_BT_2 = sum(len(wort) for wort in SPD_BT_wortliste_2) / len(SPD_BT_wortliste_2)
SPD_Schnitt_Wortlänge_BT_3 = sum(len(wort) for wort in SPD_BT_wortliste_3) / len(SPD_BT_wortliste_3)
SPD_Schnitt_Wortlänge_BT_4 = sum(len(wort) for wort in SPD_BT_wortliste_4) / len(SPD_BT_wortliste_4)
SPD_Schnitt_Wortlänge_BT_5 = sum(len(wort) for wort in SPD_BT_wortliste_5) / len(SPD_BT_wortliste_5)

Linke_Schnitt_Wortlänge_BT_1 = sum(len(wort) for wort in Linke_BT_wortliste_1) / len(Linke_BT_wortliste_1)
Linke_Schnitt_Wortlänge_BT_2 = sum(len(wort) for wort in Linke_BT_wortliste_2) / len(Linke_BT_wortliste_2)
Linke_Schnitt_Wortlänge_BT_3 = sum(len(wort) for wort in Linke_BT_wortliste_3) / len(Linke_BT_wortliste_3)
Linke_Schnitt_Wortlänge_BT_4 = sum(len(wort) for wort in Linke_BT_wortliste_4) / len(Linke_BT_wortliste_4)
Linke_Schnitt_Wortlänge_BT_5 = sum(len(wort) for wort in Linke_BT_wortliste_5) / len(Linke_BT_wortliste_5)

Grüne_Schnitt_Wortlänge_BT_1 = sum(len(wort) for wort in Grüne_BT_wortliste_1) / len(Grüne_BT_wortliste_1)
Grüne_Schnitt_Wortlänge_BT_2 = sum(len(wort) for wort in Grüne_BT_wortliste_2) / len(Grüne_BT_wortliste_2)
Grüne_Schnitt_Wortlänge_BT_3 = sum(len(wort) for wort in Grüne_BT_wortliste_3) / len(Grüne_BT_wortliste_3)
Grüne_Schnitt_Wortlänge_BT_4 = sum(len(wort) for wort in Grüne_BT_wortliste_4) / len(Grüne_BT_wortliste_4)
Grüne_Schnitt_Wortlänge_BT_5 = sum(len(wort) for wort in Grüne_BT_wortliste_5) / len(Grüne_BT_wortliste_5)

AfD_Schnitt_Wortlänge_BT_1 = sum(len(wort) for wort in AfD_BT_wortliste_1) / len(AfD_BT_wortliste_1)
AfD_Schnitt_Wortlänge_BT_2 = sum(len(wort) for wort in AfD_BT_wortliste_2) / len(AfD_BT_wortliste_2)
AfD_Schnitt_Wortlänge_BT_3 = sum(len(wort) for wort in AfD_BT_wortliste_3) / len(AfD_BT_wortliste_3)
AfD_Schnitt_Wortlänge_BT_4 = sum(len(wort) for wort in AfD_BT_wortliste_4) / len(AfD_BT_wortliste_4)
AfD_Schnitt_Wortlänge_BT_5 = sum(len(wort) for wort in AfD_BT_wortliste_5) / len(AfD_BT_wortliste_5)

FDP_Schnitt_Wortlänge_BT_1 = sum(len(wort) for wort in FDP_BT_wortliste_1) / len(FDP_BT_wortliste_1)
FDP_Schnitt_Wortlänge_BT_2 = sum(len(wort) for wort in FDP_BT_wortliste_2) / len(FDP_BT_wortliste_2)
FDP_Schnitt_Wortlänge_BT_3 = sum(len(wort) for wort in FDP_BT_wortliste_3) / len(FDP_BT_wortliste_3)
FDP_Schnitt_Wortlänge_BT_4 = sum(len(wort) for wort in FDP_BT_wortliste_4) / len(FDP_BT_wortliste_4)

**3. Teil Schachtelsätze ("Hypotaxe") bei den Bundestagsreden**

In [23]:
# Nutzt Regular Expressions, um bei . ! oder ? zu trennen
# Das [?!.] definiert eine Menge von Zeichen
CDU_sätze_BT1  = re.split(r'[?!.]', text_liste_CDU_1_str)
CDU_sätze_BT1 = [s.strip() for s in CDU_sätze_BT1 if s.strip()]
CDU_sätze_BT2  = re.split(r'[?!.]', text_liste_CDU_2_str)
CDU_sätze_BT2 = [s.strip() for s in CDU_sätze_BT2 if s.strip()]
CDU_sätze_BT3  = re.split(r'[?!.]', text_liste_CDU_3_str)
CDU_sätze_BT3 = [s.strip() for s in CDU_sätze_BT3 if s.strip()]
CDU_sätze_BT4  = re.split(r'[?!.]', text_liste_CDU_4_str)
CDU_sätze_BT4 = [s.strip() for s in CDU_sätze_BT4 if s.strip()]
CDU_sätze_BT5  = re.split(r'[?!.]', text_liste_CDU_5_str)
CDU_sätze_BT5 = [s.strip() for s in CDU_sätze_BT5 if s.strip()]

SPD_sätze_BT1  = re.split(r'[?!.]', text_liste_SPD_1_str)
SPD_sätze_BT1 = [s.strip() for s in SPD_sätze_BT1 if s.strip()]
SPD_sätze_BT2  = re.split(r'[?!.]', text_liste_SPD_2_str)
SPD_sätze_BT2 = [s.strip() for s in SPD_sätze_BT2 if s.strip()]
SPD_sätze_BT3  = re.split(r'[?!.]', text_liste_SPD_3_str)
SPD_sätze_BT3 = [s.strip() for s in SPD_sätze_BT3 if s.strip()]
SPD_sätze_BT4  = re.split(r'[?!.]', text_liste_SPD_4_str)
SPD_sätze_BT4 = [s.strip() for s in SPD_sätze_BT4 if s.strip()]
SPD_sätze_BT5  = re.split(r'[?!.]', text_liste_SPD_5_str)
SPD_sätze_BT5 = [s.strip() for s in SPD_sätze_BT5 if s.strip()]

Linke_sätze_BT1  = re.split(r'[?!.]', text_liste_Linke_1_str)
Linke_sätze_BT1 = [s.strip() for s in Linke_sätze_BT1 if s.strip()]
Linke_sätze_BT2  = re.split(r'[?!.]', text_liste_Linke_2_str)
Linke_sätze_BT2 = [s.strip() for s in Linke_sätze_BT2 if s.strip()]
Linke_sätze_BT3  = re.split(r'[?!.]', text_liste_Linke_3_str)
Linke_sätze_BT3 = [s.strip() for s in Linke_sätze_BT3 if s.strip()]
Linke_sätze_BT4  = re.split(r'[?!.]', text_liste_Linke_4_str)
Linke_sätze_BT4 = [s.strip() for s in Linke_sätze_BT4 if s.strip()]
Linke_sätze_BT5  = re.split(r'[?!.]', text_liste_Linke_5_str)
Linke_sätze_BT5 = [s.strip() for s in Linke_sätze_BT5 if s.strip()]

Grüne_sätze_BT1  = re.split(r'[?!.]', text_liste_Grüne_1_str)
Grüne_sätze_BT1 = [s.strip() for s in Grüne_sätze_BT1 if s.strip()]
Grüne_sätze_BT2  = re.split(r'[?!.]', text_liste_Grüne_2_str)
Grüne_sätze_BT2 = [s.strip() for s in Grüne_sätze_BT2 if s.strip()]
Grüne_sätze_BT3  = re.split(r'[?!.]', text_liste_Grüne_3_str)
Grüne_sätze_BT3 = [s.strip() for s in Grüne_sätze_BT3 if s.strip()]
Grüne_sätze_BT4  = re.split(r'[?!.]', text_liste_Grüne_4_str)
Grüne_sätze_BT4 = [s.strip() for s in Grüne_sätze_BT4 if s.strip()]
Grüne_sätze_BT5  = re.split(r'[?!.]', text_liste_Grüne_5_str)
Grüne_sätze_BT5 = [s.strip() for s in Grüne_sätze_BT5 if s.strip()]

AfD_sätze_BT1  = re.split(r'[?!.]', text_liste_AfD_1_str)
AfD_sätze_BT1 = [s.strip() for s in AfD_sätze_BT1 if s.strip()]
AfD_sätze_BT2  = re.split(r'[?!.]', text_liste_AfD_2_str)
AfD_sätze_BT2 = [s.strip() for s in AfD_sätze_BT2 if s.strip()]
AfD_sätze_BT3  = re.split(r'[?!.]', text_liste_AfD_3_str)
AfD_sätze_BT3 = [s.strip() for s in AfD_sätze_BT3 if s.strip()]
AfD_sätze_BT4  = re.split(r'[?!.]', text_liste_AfD_4_str)
AfD_sätze_BT4 = [s.strip() for s in AfD_sätze_BT4 if s.strip()]
AfD_sätze_BT5  = re.split(r'[?!.]', text_liste_AfD_5_str)
AfD_sätze_BT5 = [s.strip() for s in AfD_sätze_BT5 if s.strip()]

FDP_sätze_BT1  = re.split(r'[?!.]', text_liste_FDP_1_str)
FDP_sätze_BT1 = [s.strip() for s in FDP_sätze_BT1 if s.strip()]
FDP_sätze_BT2  = re.split(r'[?!.]', text_liste_FDP_2_str)
FDP_sätze_BT2 = [s.strip() for s in FDP_sätze_BT2 if s.strip()]
FDP_sätze_BT3  = re.split(r'[?!.]', text_liste_FDP_3_str)
FDP_sätze_BT3 = [s.strip() for s in FDP_sätze_BT3 if s.strip()]
FDP_sätze_BT4  = re.split(r'[?!.]', text_liste_FDP_4_str)
FDP_sätze_BT4 = [s.strip() for s in FDP_sätze_BT4 if s.strip()]

In [24]:
CDU_anzahl_schachtelsätze_BT_1 = len([s for s in CDU_sätze_BT1 if s.count(',') >= 2])
CDU_anzahl_schachtelsätze_BT_2 = len([s for s in CDU_sätze_BT2 if s.count(',') >= 2])
CDU_anzahl_schachtelsätze_BT_3 = len([s for s in CDU_sätze_BT3 if s.count(',') >= 2])
CDU_anzahl_schachtelsätze_BT_4 = len([s for s in CDU_sätze_BT4 if s.count(',') >= 2])
CDU_anzahl_schachtelsätze_BT_5 = len([s for s in CDU_sätze_BT5 if s.count(',') >= 2])

SPD_anzahl_schachtelsätze_BT_1 = len([s for s in SPD_sätze_BT1 if s.count(',') >= 2])
SPD_anzahl_schachtelsätze_BT_2 = len([s for s in SPD_sätze_BT2 if s.count(',') >= 2])
SPD_anzahl_schachtelsätze_BT_3 = len([s for s in SPD_sätze_BT3 if s.count(',') >= 2])
SPD_anzahl_schachtelsätze_BT_4 = len([s for s in SPD_sätze_BT4 if s.count(',') >= 2])
SPD_anzahl_schachtelsätze_BT_5 = len([s for s in SPD_sätze_BT5 if s.count(',') >= 2])

Linke_anzahl_schachtelsätze_BT_1 = len([s for s in Linke_sätze_BT1 if s.count(',') >= 2])
Linke_anzahl_schachtelsätze_BT_2 = len([s for s in Linke_sätze_BT2 if s.count(',') >= 2])
Linke_anzahl_schachtelsätze_BT_3 = len([s for s in Linke_sätze_BT3 if s.count(',') >= 2])
Linke_anzahl_schachtelsätze_BT_4 = len([s for s in Linke_sätze_BT4 if s.count(',') >= 2])
Linke_anzahl_schachtelsätze_BT_5 = len([s for s in Linke_sätze_BT5 if s.count(',') >= 2])

Grüne_anzahl_schachtelsätze_BT_1 = len([s for s in Grüne_sätze_BT1 if s.count(',') >= 2])
Grüne_anzahl_schachtelsätze_BT_2 = len([s for s in Grüne_sätze_BT2 if s.count(',') >= 2])
Grüne_anzahl_schachtelsätze_BT_3 = len([s for s in Grüne_sätze_BT3 if s.count(',') >= 2])
Grüne_anzahl_schachtelsätze_BT_4 = len([s for s in Grüne_sätze_BT4 if s.count(',') >= 2])
Grüne_anzahl_schachtelsätze_BT_5 = len([s for s in Grüne_sätze_BT5 if s.count(',') >= 2])

AfD_anzahl_schachtelsätze_BT_1 = len([s for s in AfD_sätze_BT1 if s.count(',') >= 2])
AfD_anzahl_schachtelsätze_BT_2 = len([s for s in AfD_sätze_BT2 if s.count(',') >= 2])
AfD_anzahl_schachtelsätze_BT_3 = len([s for s in AfD_sätze_BT3 if s.count(',') >= 2])
AfD_anzahl_schachtelsätze_BT_4 = len([s for s in AfD_sätze_BT4 if s.count(',') >= 2])
AfD_anzahl_schachtelsätze_BT_5 = len([s for s in AfD_sätze_BT5 if s.count(',') >= 2])

FDP_anzahl_schachtelsätze_BT_1 = len([s for s in FDP_sätze_BT1 if s.count(',') >= 2])
FDP_anzahl_schachtelsätze_BT_2 = len([s for s in FDP_sätze_BT2 if s.count(',') >= 2])
FDP_anzahl_schachtelsätze_BT_3 = len([s for s in FDP_sätze_BT3 if s.count(',') >= 2])
FDP_anzahl_schachtelsätze_BT_4 = len([s for s in FDP_sätze_BT4 if s.count(',') >= 2])

In [25]:
# Verhältnisermittlung
CDU_schachtel_ratio_BT1 = round(CDU_anzahl_schachtelsätze_BT_1 / (len(CDU_sätze_BT1))*100,2)
CDU_schachtel_ratio_BT2 = round(CDU_anzahl_schachtelsätze_BT_2 / (len(CDU_sätze_BT2))*100,2)
CDU_schachtel_ratio_BT3 = round(CDU_anzahl_schachtelsätze_BT_3 / (len(CDU_sätze_BT3))*100,2)
CDU_schachtel_ratio_BT4 = round(CDU_anzahl_schachtelsätze_BT_4 / (len(CDU_sätze_BT4))*100,2)
CDU_schachtel_ratio_BT5 = round(CDU_anzahl_schachtelsätze_BT_5 / (len(CDU_sätze_BT5))*100,2)

SPD_schachtel_ratio_BT1 = round(SPD_anzahl_schachtelsätze_BT_1 / (len(SPD_sätze_BT1))*100,2)
SPD_schachtel_ratio_BT2 = round(SPD_anzahl_schachtelsätze_BT_2 / (len(SPD_sätze_BT2))*100,2)
SPD_schachtel_ratio_BT3 = round(SPD_anzahl_schachtelsätze_BT_3 / (len(SPD_sätze_BT3))*100,2)
SPD_schachtel_ratio_BT4 = round(SPD_anzahl_schachtelsätze_BT_4 / (len(SPD_sätze_BT4))*100,2)
SPD_schachtel_ratio_BT5 = round(SPD_anzahl_schachtelsätze_BT_5 / (len(SPD_sätze_BT5))*100,2)

Linke_schachtel_ratio_BT1 = round(Linke_anzahl_schachtelsätze_BT_1 / (len(Linke_sätze_BT1))*100,2)
Linke_schachtel_ratio_BT2 = round(Linke_anzahl_schachtelsätze_BT_2 / (len(Linke_sätze_BT2))*100,2)
Linke_schachtel_ratio_BT3 = round(Linke_anzahl_schachtelsätze_BT_3 / (len(Linke_sätze_BT3))*100,2)
Linke_schachtel_ratio_BT4 = round(Linke_anzahl_schachtelsätze_BT_4 / (len(Linke_sätze_BT4))*100,2)
Linke_schachtel_ratio_BT5 = round(Linke_anzahl_schachtelsätze_BT_5 / (len(Linke_sätze_BT5))*100,2)

Grüne_schachtel_ratio_BT1 = round(Grüne_anzahl_schachtelsätze_BT_1 / (len(Grüne_sätze_BT1))*100,2)
Grüne_schachtel_ratio_BT2 = round(Grüne_anzahl_schachtelsätze_BT_2 / (len(Grüne_sätze_BT2))*100,2)
Grüne_schachtel_ratio_BT3 = round(Grüne_anzahl_schachtelsätze_BT_3 / (len(Grüne_sätze_BT3))*100,2)
Grüne_schachtel_ratio_BT4 = round(Grüne_anzahl_schachtelsätze_BT_4 / (len(Grüne_sätze_BT4))*100,2)
Grüne_schachtel_ratio_BT5 = round(Grüne_anzahl_schachtelsätze_BT_5 / (len(Grüne_sätze_BT5))*100,2)

AfD_schachtel_ratio_BT1 = round(AfD_anzahl_schachtelsätze_BT_1 / (len(AfD_sätze_BT1))*100,2)
AfD_schachtel_ratio_BT2 = round(AfD_anzahl_schachtelsätze_BT_2 / (len(AfD_sätze_BT2))*100,2)
AfD_schachtel_ratio_BT3 = round(AfD_anzahl_schachtelsätze_BT_3 / (len(AfD_sätze_BT3))*100,2)
AfD_schachtel_ratio_BT4 = round(AfD_anzahl_schachtelsätze_BT_4 / (len(AfD_sätze_BT4))*100,2)
AfD_schachtel_ratio_BT5 = round(AfD_anzahl_schachtelsätze_BT_5 / (len(AfD_sätze_BT5))*100,2)

FDP_schachtel_ratio_BT1 = round(FDP_anzahl_schachtelsätze_BT_1 / (len(FDP_sätze_BT1))*100,2)
FDP_schachtel_ratio_BT2 = round(FDP_anzahl_schachtelsätze_BT_2 / (len(FDP_sätze_BT2))*100,2)
FDP_schachtel_ratio_BT3 = round(FDP_anzahl_schachtelsätze_BT_3 / (len(FDP_sätze_BT3))*100,2)
FDP_schachtel_ratio_BT4 = round(FDP_anzahl_schachtelsätze_BT_4 / (len(FDP_sätze_BT4))*100,2)

In [26]:
df_woerter = pd.DataFrame({
    ("Programm", "Ø WL"): [CDU_Schnitt_Wortlänge_WP, SPD_Schnitt_Wortlänge_WP, Linke_Schnitt_Wortlänge_WP, AfD_Schnitt_Wortlänge_WP, Grüne_Schnitt_Wortlänge_WP, FDP_Schnitt_Wortlänge_WP],
    ("Programm", "% >= 14 Zeichen"): [CDU_14er_ratio, SPD_14er_ratio, Linke_14er_ratio, AfD_14er_ratio, Grüne_14er_ratio, FDP_14er_ratio],
    ("Periode 1", "Ø WL"): [CDU_Schnitt_Wortlänge_BT_1, SPD_Schnitt_Wortlänge_BT_1, Linke_Schnitt_Wortlänge_BT_1, AfD_Schnitt_Wortlänge_BT_1, Grüne_Schnitt_Wortlänge_BT_1, FDP_Schnitt_Wortlänge_BT_1],
    ("Periode 1", "% >= 14 Zeichen"): [CDU_14er_ratio_BT_1, SPD_14er_ratio_BT_1, Linke_14er_ratio_BT_2, AfD_14er_ratio_BT_1, Grüne_14er_ratio_BT_1, FDP_14er_ratio_BT_1],
    ("Periode 2", "Ø WL"): [CDU_Schnitt_Wortlänge_BT_2, SPD_Schnitt_Wortlänge_BT_2, Linke_Schnitt_Wortlänge_BT_2, AfD_Schnitt_Wortlänge_BT_2, Grüne_Schnitt_Wortlänge_BT_2, FDP_Schnitt_Wortlänge_BT_2],
    ("Periode 2", "% >= 14 Zeichen"): [CDU_14er_ratio_BT_2, SPD_14er_ratio_BT_2, Linke_14er_ratio_BT_2, AfD_14er_ratio_BT_2, Grüne_14er_ratio_BT_2, FDP_14er_ratio_BT_2],
    ("Periode 3", "Ø WL"): [CDU_Schnitt_Wortlänge_BT_3, SPD_Schnitt_Wortlänge_BT_3, Linke_Schnitt_Wortlänge_BT_3, AfD_Schnitt_Wortlänge_BT_3, Grüne_Schnitt_Wortlänge_BT_3, FDP_Schnitt_Wortlänge_BT_3],
    ("Periode 3", "% >= 14 Zeichen"): [CDU_Schnitt_Wortlänge_BT_4, SPD_Schnitt_Wortlänge_BT_4, Linke_Schnitt_Wortlänge_BT_4, AfD_Schnitt_Wortlänge_BT_4, Grüne_Schnitt_Wortlänge_BT_4, FDP_Schnitt_Wortlänge_BT_4],
    ("Periode 4", "Ø WL"): [CDU_Schnitt_Wortlänge_BT_4, SPD_Schnitt_Wortlänge_BT_4, Linke_Schnitt_Wortlänge_BT_4, AfD_Schnitt_Wortlänge_BT_4, Grüne_Schnitt_Wortlänge_BT_4, FDP_Schnitt_Wortlänge_BT_4],
    ("Periode 4", "% >= 14 Zeichen"): [CDU_14er_ratio_BT_4, SPD_14er_ratio_BT_4, Linke_14er_ratio_BT_4, AfD_14er_ratio_BT_4, Grüne_14er_ratio_BT_4, FDP_14er_ratio_BT_4],
    ("Periode 5", "Ø WL"): [CDU_Schnitt_Wortlänge_BT_5, SPD_Schnitt_Wortlänge_BT_5, Linke_Schnitt_Wortlänge_BT_5, AfD_Schnitt_Wortlänge_BT_5, Grüne_Schnitt_Wortlänge_BT_5,0.0],
    ("Periode 5", "% >= 14 Zeichen"): [CDU_14er_ratio_BT_5, SPD_14er_ratio_BT_5, Linke_14er_ratio_BT_5, AfD_14er_ratio_BT_5, Grüne_14er_ratio_BT_5,0.0],
}).round(1)

df_woerter.index = ["CDU/CSU", "SPD", "Linke", "AfD", "Grüne", "FDP"]
df_woerter = df_woerter.replace(0.0, "-")
df_woerter

Programm                 Periode 1                 Periode 2  \
            Ø WL % >= 14 Zeichen      Ø WL % >= 14 Zeichen      Ø WL   
CDU/CSU      6.8             9.0       5.8             5.0       5.9   
SPD          6.7             8.3       5.9             5.0       5.9   
Linke        6.8             8.4       5.9             5.6       6.0   
AfD          7.0             9.5       5.9             4.9       5.9   
Grüne        6.8             8.4       5.9             4.9       5.9   
FDP          6.9             9.0       5.9             5.0       5.9   

                        Periode 3                 Periode 4                  \
        % >= 14 Zeichen      Ø WL % >= 14 Zeichen      Ø WL % >= 14 Zeichen   
CDU/CSU             4.9       5.8             5.8       5.8             4.4   
SPD                 4.9       5.8             5.7       5.7             4.1   
Linke               5.6       5.9             5.7       5.7             3.9   
AfD                 5.0       5.9             5.8       5.8             4.4   
Grüne               5.0       5.8             5.6       5.6             3.8   
FDP                 5.1       5.9             5.9       5.9             4.8   

        Periode 5                  
             Ø WL % >= 14 Zeichen  
CDU/CSU       5.9             4.9  
SPD           5.9             4.8  
Linke         5.9             4.8  
AfD           6.0             5.0  
Grüne         5.7             4.4  
FDP             -               -

In [27]:
df_hypotaxe = pd.DataFrame({
    ("Programm", "% Hypo-taxe"): [CDU_schachtel_ratio, SPD_schachtel_ratio, Linke_schachtel_ratio, AfD_schachtel_ratio, Grüne_schachtel_ratio, FDP_schachtel_ratio],
    ("Periode 1", "% Hypo-taxe"): [CDU_schachtel_ratio_BT1, SPD_schachtel_ratio_BT1, Linke_schachtel_ratio_BT1, AfD_schachtel_ratio_BT1, Grüne_schachtel_ratio_BT1, FDP_schachtel_ratio_BT1],
    ("Periode 2", "% Hypo-taxe"): [CDU_schachtel_ratio_BT2, SPD_schachtel_ratio_BT2, Linke_schachtel_ratio_BT2, AfD_schachtel_ratio_BT2, Grüne_schachtel_ratio_BT2, FDP_schachtel_ratio_BT2],
    ("Periode 3", "% Hypo-taxe"): [CDU_schachtel_ratio_BT3, SPD_schachtel_ratio_BT3, Linke_schachtel_ratio_BT3, AfD_schachtel_ratio_BT3, Grüne_schachtel_ratio_BT3, FDP_schachtel_ratio_BT3],
    ("Periode 4", "% Hypo-taxee"): [CDU_schachtel_ratio_BT4, SPD_schachtel_ratio_BT4, Linke_schachtel_ratio_BT4, AfD_schachtel_ratio_BT4, Grüne_schachtel_ratio_BT4, FDP_schachtel_ratio_BT4],
    ("Periode 5", "% Hypo-taxe"): [CDU_schachtel_ratio_BT5, SPD_schachtel_ratio_BT5, Linke_schachtel_ratio_BT5, AfD_schachtel_ratio_BT5, Grüne_schachtel_ratio_BT5,0.0],
}).round(2)

df_hypotaxe.index = ["CDU/CSU", "SPD", "Linke", "AfD", "Grüne", "FDP"]
df_hypotaxe = df_hypotaxe.replace(0.0, "-")
df_hypotaxe

,Programm,Periode 1,Periode 2,Periode 3,Periode 4,Periode 5
,% Hypo-taxe,% Hypo-taxe,% Hypo-taxe,% Hypo-taxe,% Hypo-taxee,% Hypo-taxe
CDU/CSU,9.29,28.06,26.60,28.15,29.58,27.17
SPD,17.69,28.44,27.72,27.73,30.40,28.02
Linke,17.80,23.60,22.86,23.40,25.39,25.66
AfD,17.18,26.38,24.86,25.79,24.74,23.84
Grüne,20.72,29.91,27.74,28.51,32.60,28.61
FDP,13.59,28.20,28.29,27.08,27.17,-


In [28]:
# in LaTex überführen

latex_code = df_woerter.to_latex(index=False, caption='Reden', label='tab:meine_tabelle')
with open('tabelle_woerter.tex', 'w') as f:
    f.write(latex_code)

In [29]:
# in LaTex überführen

latex_code = df_hypotaxe.to_latex(index=False, caption='Reden', label='tab:meine_tabelle')
with open('tabelle_hypotaxe.tex', 'w') as f:
    f.write(latex_code)